© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# Machine Translation (50 points)




**Import the notebook into Colab and work in it there!**

**Contestant's name:**


Stux, but instead of him, Intelligent John has now become a true Parisian. During his tour of France he collected French expressions, sentences and personal names in various languages in his notebook.
Since he has not yet fully mastered the French language, he calls on a computer translator for help so that he can understand his notes. Create such a program for him!

---
## Preparations

**Guides to the tools needed to solve the task:**
1. [Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
2. [Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
3. [Tokenization](https://medium.com/@utkarsh.kant/tokenization-a-complete-guide-3f2dd56c0682)
4. [Tokenization, Mapping and Padding](https://medium.com/@lokaregns/preparing-text-data-for-transformers-tokenization-mapping-and-padding-9fbfbce28028)
5. [PyTorch Training/Inference](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html)
6. [PyTorch Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
7. [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
8. [Matplotlib Bar Plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html)
9. [LSTM, GRU](https://medium.com/@mervebdurna/nlp-with-deep-learning-neural-networks-rnns-lstms-and-gru-3de7289bb4f8)
10. [GRU PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html)
11. [Next Word Prediction](https://www.geeksforgeeks.org/next-word-prediction-with-deep-learning-in-nlp/)



⚠️ **The data is no longer available.** The language dataset + Encoder/Decoder model weights, unique competition material, were lost because the Google Drive was deleted.

The original Google Drive links do not work.

In [ ]:
# @title Installing Dependencies
!pip install pandas --quiet
!pip install torchtext --quiet

In [ ]:
# ⚠️ Data not available (Google Drive deleted)
# @title Downloading the Language Dataset and the Model Weights
# !gdown -qq 16p4Ky4RyD8o_6wCp0VV3KukVmmOUPvV4
# !gdown -qq 1vmBeAjTr7mK4mI-8h7UkkII5NWSd2TT9
# !gdown -qq 16O8wQ0-LYrANbYVuFGnX7epiIB2zIqcY
# !unzip -qq nyelvek.zip -d .
# !rm -rf nyelvek.zip


### Required Libraries

We have imported a few libraries to get you started, but feel free to use any PyTorch-based tool if needed. Please note that Keras and TensorFlow are **NOT ALLOWED** for solving this task!

In [ ]:
import io
import re
import math
import random
import unicodedata

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

from tqdm.notebook import tqdm
from sklearn.utils import shuffle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Task 1: Language Representation (10 Points)

In the first task we want you to get familiar with the dataset and to build a language representation. Your tasks are the following:

#### 1. Print the first 10 sentence pairs of the dataset (.txt file)! [Pandas Read Text File](https://www.geeksforgeeks.org/how-to-read-text-files-with-pandas/) (2 points)

#### 2. Create a **Lang** class whose purpose is to manage the vocabulary of a language and to convert text data into the **numeric formats** required by machine learning models! (4 points)

 The structure of the Lang class should be the following:

- **Attributes**:
  - `name`: A string denoting the name of the language.
  - `word2index`: A dictionary that maps words to their unique indices.
  - `word2count`: A dictionary for counting the occurrences of each word.
  - `index2word`: A dictionary that maps indices to words, pre-initialized with the “SOS” (start of sentence), “EOS” (end of sentence) and “PAD” (padding) tokens.
  - `n_words`: The integer representation of the unique words, **starting from 3**, in order to account for the special (SOS, EOS, PAD) tokens.
- **Methods**:
  - `addSentence(sentence)`: Splits the sentence into words and processes each word using `addWord`.
  - `addWord(word)`: Adds a word to the dictionaries, updating all relevant attributes.

#### 3. In addition to defining the class, we will also need the following helper functions. Two of them we have already written for you as a help. Implement the `read_langs` function! (4 points)

- `unicodeToAscii(s)`: Converts a Unicode string into a plain ASCII string. **(GIVEN)**
  
- `normalizeString(s)`: Prepares the string data by cleaning and standardizing it into a format suitable for processing. **(GIVEN)**

- `read_langs(lang1, lang2, reverse)`
    - **Parameters**:
      - `lang1`, `lang2`: The names of the two languages appearing in the data file.
      - `reverse`: Boolean indicating whether the order of the language pairs should be reversed during processing.
    - **Functionality**:
      - Reading in a given file that contains paired sentences of two languages.
      - Normalizing the sentences using `normalizeString`.
      - Creating instances of the `Lang` class for vocabulary management.
      - Optionally reversing the language pairs if specified as a parameter.

In [ ]:
class Lang():
    def __init__(self, name):
        raise NotImplementedError("This function has not been implemented yet.")

    def add_sentence(self, sentence):
        raise NotImplementedError("This function has not been implemented yet.")

    def add_word(self, word):
        raise NotImplementedError("This function has not been implemented yet.")


def unicode_to_ascii(s):
  return ''.join(
      c for c in unicodedata.normalize('NFD', s)
      if unicodedata.category(c) != 'Mn')

def normalize_string(s):
  s = unicode_to_ascii(s.lower().strip())
  s = re.sub(r"([.!?])", r" \1", s)
  s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
  return s


def read_langs(lang1, lang2, reverse=False):
    raise NotImplementedError("This function has not been implemented yet.")

# Task 2: Data Cleaning (15 points)

During data cleaning we want to filter out sentences that represent common conversational expressions. To help with this, we have created an `eng_prefixes` variable that contains the opening sentence fragments by which we want to filter the whole dataset.

#### 1. Instantiate a French and an English Language using the `read_langs` function, with the language pairs reversed! (French-English) (3 points)

#### 2. Filter out from the English language those sentences that do not begin with the words found in the `eng_prefixes` **tuple**! (3 points)

#### 3. Also filter out from the English language those sentences where the number of words in the sentence exceeds 10! (`MAX_LENGTH`) (3 points)

#### 4. Update the attributes of the two instantiated languages using the filtered sentences and the `add_senteces` function! (3 points)

#### 5. Implement the `prepare_data` function, which will return the two language objects and the filtered sentence pairs in one big list! (3 points)

In [ ]:
MAX_LENGTH = 10
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


def filter_pairs():
    raise NotImplementedError("This function has not been implemented yet.")


def prepare_data():
    raise NotImplementedError("This function has not been implemented yet.")

# Task 3: Word Distribution (6 points)

During data cleaning we want to filter out sentences that represent common conversational expressions. To help with this, we have created an `eng_prefixes` variable that contains the opening sentence fragments by which we want to filter the whole dataset.

#### 1. Calculate what percentage of the words makes up 80% of the total text corpus, for both the English and the French language! (3 points)

#### 2. Create two Bar Plots that visualize the 100 most used English and the most used French words! (3 points) [Matplotlib Bar Plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html)

In [ ]:
def plot_lang():
    raise NotImplementedError("This function has not been implemented yet.")

# Task 4: RNN Architecture (12 points)

Our goal is to create a network that takes an input sentence in one language and then produces the translation of the sentence in an output language. Our network will use an RNN consisting of an encoder and a decoder. The encoder first converts our input sentence into a vector, and passes this compressed vector to the decoder, which translates the text into the given language. The figure below illustrates this process in more detail:

<img src="https://raw.githubusercontent.com/NeuromatchAcademy/course-content-dl/main/projects/static/seq2seq.png" width="600" height="300">

[LSTM, GRU](https://medium.com/@mervebdurna/nlp-with-deep-learning-neural-networks-rnns-lstms-and-gru-3de7289bb4f8)

We want to build a model that can generate an English translation from the French language. As a help, we have already created the architecture of the decoder, so your task will be to build the architecture of the encoder. Your tasks are the following:

#### 1. Create the architecture of the encoder based on the figure! (4 points) [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)

#### 2. Instantiate the two models, then load the weights of the encoder and the decoder (`decoder.pth`, `encoder.pth`)! (1 point) ([Pytorch Loading Weights](https://pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html#:~:text=To%20load%20model%20weights%2C%20you,parameters%20using%20load_state_dict()%20method.&text=be%20sure%20to%20call%20model,normalization%20layers%20to%20evaluation%20mode))

#### 3. Convert the language sequences into the appropriate format as follows: (7 points)

  * For each input sentence, convert the words to their corresponding indices, insert a start marker (`SOS`), an end marker (`EOS`), and make sure that enough padding tokens (`PAD`) are inserted in front of the sequence so that the uniform length is maintained (`MAX_LENGTH`)! We represent the input sentence with left padding, since the GRU network processes the sentence from left to right, and we want the output to be as close as possible to our sentence. [GRU PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html)
  * For each output sentence, start with a start token (`SOS`), followed by the indices of the individual words! Append an end token (`EOS`) to it, and close it with padding tokens (`PAD`) up to `MAX_LENGTH`! Unlike with the GRU, use right padding for the partial translation sentence, because we want our decoder to process our context immediately. Finally, our target sequence is our partial translation shifted by one to the left (`next word prediction`, [Next Word Prediction](https://www.geeksforgeeks.org/next-word-prediction-with-deep-learning-in-nlp/)).

  * The target sequence must be produced from the output sequence by removing the start token `SOS` and shifting the sequence, so that the model can predict the next word in the sequence. In order to preserve the length, append `PAD` tokens at the end up to `MAX_LENGTH`!

Architecture of the encoder:

<a href="https://ibb.co/LNgtxX6"><img src="https://i.ibb.co/vXdP36m/encoder.png" alt="encoder" border="0"></a>

Architecture of the decoder:


<a href="https://ibb.co/1qWysVj"><img src="https://i.ibb.co/wWmGJHj/decoder.png" alt="decoder" border="0"></a>

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        raise NotImplementedError("This function has not been implemented yet.")

    def forward(self, input, hidden):
        raise NotImplementedError("This function has not been implemented yet.")

    def initHidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        raise NotImplementedError("This function has not been implemented yet.")

    def forward(self, input, hidden):
        raise NotImplementedError("This function has not been implemented yet.")

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:
def prepare_data(input_lang, output_lang, pairs, max_len=MAX_LENGTH+2):
    x_input = []
    x_output = []
    target = []
    return x_input, x_output, target

# 5\. Task: Generating Translations (7 points)

As a final task, we would like to test the capabilities of our model. Your tasks are the following:

#### 1. Generate tokens using the `predict` function! (1 point)

#### 2. Translate the first 10 French sentences from the French-English sentence pair list into English. (6 points)

Generating a complete text is a very simple iterative process:

* Initializing the output sentence means just a single `SOS` token, and then the following iterative process must be followed:
    1. Predict the probability distribution of the next token based on the partial output sentence! (`Softmax`, appearing in the architecture)
    2. Choose the most likely token!
    3. Append this token to the output sentence!
    4. If the resulting token is `EOS`, exit the loop!

In [ ]:
def predict(encoder, decoder, input, output):
  _, hidden = encoder(input, encoder.initHidden(input.shape[0]))
  out, _ = decoder(output, hidden)
  return out

In [ ]:
def gen_translation(encoder, decoder, text, input_lang, output_lang, max_len=MAX_LENGTH+2):
    raise NotImplementedError("This function has not been implemented yet.")

def predict_first_ten():
    raise NotImplementedError("This function has not been implemented yet.")

---
**You have reached the end of the task**. Download the finished Notebook as follows:
```
File → Download → Download .ipynb
```
then upload it to the CMS system **together with the other solutions**, packaged in an archive **(.zip)**.